# 03 – Privacy & Governance Compliance Audit

**Role:** Governance Officer  
**Task Force:** NovaCred  

## 1. Executive Summary & Scope
As the Governance Task Force, our objective is to audit the NovaCred credit application ecosystem. While the Data Engineering and Data Science teams focused on data quality and algorithmic fairness (`01-data-quality.ipynb` and `02-bias-analysis.ipynb`), this document provides a comprehensive **Regulatory Compliance Audit**.

This audit evaluates the system against the **General Data Protection Regulation (GDPR)** and the **EU Artificial Intelligence Act (AI Act)**. We identify compliance gaps, demonstrate technical privacy safeguards, and outline a mandatory governance roadmap.

## 2. Governance Risk Assessment Matrix

Based on the technical audits, we have mapped the identified legal and ethical violations onto a Risk Matrix to prioritize remediation for the CTO.

In [6]:
import pandas as pd

risk_data = {
    'Identified Issue': ['SSN in Plain Text', 'Gender & Age Bias (DIR < 0.8)', 'Alcohol Category Collection', 'IP Address Storage', 'Lack of Human Oversight'],
    'Regulatory Framework': ['GDPR Art. 32 (Security)', 'EU AI Act Art. 10 (Data Bias)', 'GDPR Art. 9 (Sensitive Data)', 'GDPR Art. 5 (Minimization)', 'AI Act Art. 14 / GDPR Art. 22'],
    'Impact (1-5)': [5, 4, 5, 2, 4],
    'Likelihood (1-5)': [4, 5, 4, 5, 5],
    'Risk Level': ['Critical', 'High', 'Critical', 'Low', 'High']
}

risk_df = pd.DataFrame(risk_data)
print("NovaCred Governance Risk Matrix")

def color_risk(val):
    # Define background color based on the risk level
    if val == 'Critical':
        return 'background-color: #ff9999; color: black; font-weight: bold'
    elif val == 'High':
        return 'background-color: #ffcc99; color: black; font-weight: bold'
    return ''

try:
    # Handle Pandas version compatibility (map vs applymap)
    if hasattr(risk_df.style, 'map'):
        styled_df = risk_df.style.map(color_risk, subset=['Risk Level'])
    else:
        styled_df = risk_df.style.applymap(color_risk, subset=['Risk Level'])
    display(styled_df)
except Exception as e:
    # Fallback in case jinja2 is missing or other styling errors occur
    print(f"Note: Styling error ({e}). Displaying standard table.")
    display(risk_df)

NovaCred Governance Risk Matrix


,Identified Issue,Regulatory Framework,Impact (1-5),Likelihood (1-5),Risk Level
0,SSN in Plain Text,GDPR Art. 32 (Security),5,4,Critical
1,Gender & Age Bias (DIR < 0.8),EU AI Act Art. 10 (Data Bias),4,5,High
2,Alcohol Category Collection,GDPR Art. 9 (Sensitive Data),5,4,Critical
3,IP Address Storage,GDPR Art. 5 (Minimization),2,5,Low
4,Lack of Human Oversight,AI Act Art. 14 / GDPR Art. 22,4,5,High


## 3. PII Inventory & Lawful Basis Mapping (GDPR Art. 6 & 9)
To govern data effectively, we must map what we possess. Below is the programmatic PII inventory classifying the sensitivity of the fields found in `raw_credit_applications.json`.

### 3.1 PII Catalog
Programmatically classify every field in the dataset by sensitivity level and map it to its GDPR lawful basis and required safeguard.

In [7]:
# Building the Data Governance Catalog
governance_catalog = pd.DataFrame({
    'Field_Name': ['full_name', 'email', 'ssn', 'ip_address', 'date_of_birth', 'zip_code', 'spending_behavior'],
    'Identifier_Type': ['Direct', 'Direct', 'Direct (Critical)', 'Quasi-Identifier', 'Quasi-Identifier', 'Quasi-Identifier', 'Behavioral/Sensitive'],
    'GDPR_Lawful_Basis': ['Art. 6(1)(b) Contract', 'Art. 6(1)(b) Contract', 'Art. 6(1)(b) Contract', 'None (Violation)', 'Art. 6(1)(b) Contract', 'None (Violation)', 'Art. 9 Explicit Consent Req.'],
    'Re_identification_Risk': ['Low', 'Low', 'High', 'Medium', 'High (if combined)', 'High (Proxy Risk)', 'Medium']
})

print("NovaCred Data Governance Catalog")
display(governance_catalog)

NovaCred Data Governance Catalog


,Field_Name,Identifier_Type,GDPR_Lawful_Basis,Re_identification_Risk
0,full_name,Direct,Art. 6(1)(b) Contract,Low
1,email,Direct,Art. 6(1)(b) Contract,Low
2,ssn,Direct (Critical),Art. 6(1)(b) Contract,High
3,ip_address,Quasi-Identifier,None (Violation),Medium
4,date_of_birth,Quasi-Identifier,Art. 6(1)(b) Contract,High (if combined)
5,zip_code,Quasi-Identifier,None (Violation),High (Proxy Risk)
6,spending_behavior,Behavioral/Sensitive,Art. 9 Explicit Consent Req.,Medium


The catalog confirms that 5 fields contain direct identifiers (full name, email, SSN, IP address, date of birth) with no pseudonymisation applied in the raw dataset. SSN and date of birth are classified as high-sensitivity with no documented lawful basis for collection beyond creditworthiness assessment.

### 3.2 Critical Violations Identified
- **Article 5(1)(c) - Data Minimization:** The collection of `ip_address` provides no legitimate value for establishing creditworthiness.
- **Article 9 - Special Categories of Data:** The `spending_behavior` array contains raw transaction tags, including **"Alcohol"** (e.g., in `app_200`). Tracking such data infers health status and potential addictions. Processing this without explicit consent is a severe GDPR breach.
- **Fairness and Proxy Discrimination:** As confirmed in `02-bias-analysis.ipynb`, `zip_code` serves as a geographic proxy for protected attributes, enabling indirect discrimination.

## 4. Technical Safeguards: Privacy by Design (Art. 25 GDPR)

We must engineer privacy into the ingestion pipeline (Raw to Analytics Zone). We apply:
1. **Data Minimization:** Dropping unnecessary fields.
2. **Pseudonymization:** Replacing direct identifiers with a surrogate value (Hashing). The data remains subject to GDPR but security is vastly improved.

In [8]:
import hashlib
import numpy as np

# Simulation of a batch of raw ingested data
raw_batch = pd.DataFrame({
    'app_id': ['app_200', 'app_306', 'app_163'],
    'full_name': ['Jerry Smith', 'Anna White', 'Edward White'],
    'ssn': ['596-64-4340', '757-27-8131', '684-21-9902'],
    'ip_address': ['192.168.48.155', '10.26.176.56', '172.16.254.1']
})

def secure_ingestion_pipeline(df):
    """Applies automated privacy controls to raw data."""
    df_secure = df.copy()
    
    # 1. Data Minimization (IP removal)
    df_secure = df_secure.drop(columns=['ip_address'])
    
    # 2. Pseudonymization (SSN hashing)
    def hash_identifier(val):
        if pd.isna(val): return np.nan
        return hashlib.sha256(str(val).encode('utf-8')).hexdigest()[:16] # Truncated for display
        
    df_secure['ssn_token'] = df_secure['ssn'].apply(hash_identifier)
    df_secure = df_secure.drop(columns=['ssn']) # Destruction of the plain text data
    
    return df_secure

secured_data = secure_ingestion_pipeline(raw_batch)
print("Analytics Zone: Secured Data")
display(secured_data)

Analytics Zone: Secured Data


,app_id,full_name,ssn_token
0,app_200,Jerry Smith,2caf30528c21a10e
1,app_306,Anna White,618befc68f8bfb7b
2,app_163,Edward White,338011da6352b21b


## 5. EU AI Act Classification & Actionable Roadmap

The NovaCred Machine Learning model evaluates creditworthiness. According to **Annex III, Point 5(b)** of the **EU AI Act**, this automatically classifies it as a **High-Risk AI System**. 

To avoid severe regulatory penalties, we propose the following mandatory Governance Roadmap:

### 5.1 Process & Policy Controls
1. **Human Oversight (AI Act Art. 14 & GDPR Art. 22):** Automated rejections based purely on `algorithm_risk_score` are unlawful. We must institute a **Credit Review Committee** (Human-in-the-Loop) to allow applicants to appeal automated rejections.
2. **Transparency (AI Act Art. 13):** The system must generate human-readable explanations. `rejection_reason: "algorithm_risk_score"` violates transparency mandates.
3. **Data Retention Policy (GDPR Art. 5(1)(e)):** Implement an automated data lifecycle rule. Rejected applications must be purged after 5 years.

### 5.2 Technical & Algorithmic Controls
1. **Bias-Gate Deployment (AI Act Art. 10):** We must establish a CI/CD pipeline check. If the **Disparate Impact Ratio (DIR)** for any protected attribute falls below 0.8, model deployment is blocked.
2. **Feature Pruning:** Remove `zip_code` from the training features to eliminate geographic proxy discrimination.
3. **Audit Trails (AI Act Art. 12):** Implement immutable logging mechanisms that record the model version, input features, and decision timestamps.